In [1]:
#Use the following before running code
!python -m spacy download en_core_web_sm

zsh:1: command not found: python


In [2]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from spacy import load
from csv import reader

def readFromCSV(path: str):
    data = None
    with open(path, 'r', encoding='UTF-8') as f:
        data = list(reader(f))
    return data

def preprocess(sents: list[str]):
    nlp = load("en_core_web_sm", disable=["ner", "parser"])
    out = []
    for sent in sents:
        sent = ' '.join(sent.split())
        doc = nlp(sent)
        out.append(' '.join(token.lemma_ for token in doc if not token.is_punct)) # keep stopwords as 'not' can sometimes be a stopword and ruin our sentiment analysis
    return out

def main():
    # Process labeled data
    labeledData = readFromCSV('data/labeledData.csv')[1:]
    unlabeledData = readFromCSV('data/unlabeledData.csv')[1:]

    labeledEmailMessages = preprocess([line[0] for line in labeledData])
    # labeledRejectionOutcomes = [0 if line[1] == 'reject' else 1 for line in labeledData]
    labeledRejectionOutcomes = [line[1] for line in labeledData]
    unlabeledEmailMessages = preprocess([line[9] for line in unlabeledData])
    #print(unlabeledEmailMessages)

    bowVectorizer = CountVectorizer(max_features=5000)

    X = bowVectorizer.fit_transform(labeledEmailMessages) #[line[0] for line in labeledData] is a list of strings
    y = labeledRejectionOutcomes

    # print(X.get_shape())
    # print(len(y))

    X_train, X_dev, y_train, y_dev = train_test_split(X, y)

    model = ComplementNB().fit(X_train, y_train)
   
    #print("Training F1")
    y_pred_dev = model.predict(X_dev)
    f1 = f1_score(y_dev, y_pred_dev, average='micro')
    print(f"F1 score: {f1:.4f}\n")

    #print("Test Results")
    X_test = bowVectorizer.transform(unlabeledEmailMessages)
    y_pred_test = model.predict(X_test)
    #print(*[reject + ' ' + message for message, reject in zip(unlabeledEmailMessages, y_pred_test)], sep='\n')

if __name__ == '__main__':
    main()


F1 score: 0.8182



In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import joblib

def main(save_model=False):
    # Process labeled data
    labeledData = [line for line in readFromCSV('temp-data/labeled-data.csv')[1:] if line[1] != 'irrelevant'] 
    # not including the irrelevent emails in the training data for now?
    unlabeledData = readFromCSV('temp-data/unlabeled-data.csv')[1:]
    labeledEmailMessages = preprocess([line[0] for line in labeledData])
    labeledRejectionOutcomes = [line[1] for line in labeledData]
    unlabeledEmailMessages = preprocess([line[1] for line in unlabeledData])
    #print(unlabeledEmailMessages)

    bowVectorizer = CountVectorizer(max_features=5000)

    X = bowVectorizer.fit_transform(labeledEmailMessages) #[line[0] for line in labeledData] is a list of strings
    y = labeledRejectionOutcomes

    X_train, X_dev, y_train, y_dev = train_test_split(X, y, random_state=42, stratify=y)

    model = ComplementNB().fit(X_train, y_train)

    if save_model:
        joblib.dump(model, 'model.joblib')
        joblib.dump(bowVectorizer, 'vectorizer.joblib')
   
    #print("Training F1")
    y_pred_dev = model.predict(X_dev)
    f1 = f1_score(y_dev, y_pred_dev, average='macro')
    print(f"F1 score: {f1:.4f}\n")


    print("Test Results")
    X_test = bowVectorizer.transform(unlabeledEmailMessages)
    y_pred_test = model.predict(X_test)
    print(*[reject + ': ' + message for message, reject in zip(unlabeledEmailMessages, y_pred_test)], sep='\n')

    print("\nClassification Report:", classification_report(y_dev, y_pred_dev))
    print("\nConfusion Matrix: ", confusion_matrix(y_dev, y_pred_dev))
    print("\nAccuracy Score: ", f"{accuracy_score(y_dev, y_pred_dev):.2f}")

if __name__ == '__main__':
    main(save_model=True)

# run with save_model=True to save the model and vectorizer for later use
# run with save_model=False to just evaluate the model without saving

In [ ]:
# hugging face
!pip install huggingface_hub -q

from huggingface_hub import login
login()

from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id="rhallett/email-rejection-classifier", repo_type="model", exist_ok = True)

api.upload_file()